In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("🔄 Loading Qwen model...")
model_name = "Qwen/Qwen2-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ Model loaded successfully!")
print(f"📍 Device: {next(model.parameters()).device}")

🔄 Loading Qwen model...


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Model loaded successfully!
📍 Device: cuda:0


In [3]:
def ask_qwen(prompt, system_prompt="You are a helpful IT assistant.", max_new_tokens=300):
    """Core function to call Qwen — we'll reuse this everywhere"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Only return the new tokens (not the input)
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

# Test it
response = ask_qwen("Say hello and confirm you are working.")
print("Qwen says:", response)

Qwen says: Hello! I am here to assist you with any questions or concerns you may have related to technology. Please feel free to ask me anything!
